# Digital Forge Reel — Colab Renderer

Render the HTML/CSS/JS scene into a vertical MP4 video directly on Google Colab.

**Recommended runtime:** GPU (Runtime → Change runtime type → T4 GPU)

## What this notebook does
1. Unzips the project
2. Runs the one-shot setup script (installs Node 20, fonts, ffmpeg, Playwright Chromium, Python deps)
3. Renders the English + Arabic videos (GPU capture + NVENC encoding)
4. Downloads them to your machine / Google Drive

## Important notes
- **Use GPU runtime** (Runtime → Change runtime type → T4 GPU) for ~5-10x speedup
- We do NOT use `apt install chromium-browser` — that's a broken snap stub on Colab. Playwright downloads its own real Chromium.
- The `--gpu` flag enables Chrome GPU rasterization (faster capture)
- The `--encoder nvenc` flag uses NVIDIA hardware encoding (faster encode)
- **Resume support**: if a render is interrupted or fails mid-encode, just re-run the same cell — it skips already-captured frames and retries the encode

## Step 1 — Upload the project zip

Upload `digital-forge-reel.zip` to Colab using the file browser on the left (or mount Drive).

Then run the cell below to unzip it.

In [ ]:
# Unzip the project
%cd /content
!unzip -o digital-forge-reel.zip -d /content/
%cd /content/digital-forge-reel
!ls -la

## Step 2 — Run the one-shot setup script

This installs everything: Node 20, fonts, ffmpeg, Python deps, Playwright Chromium binary.

Takes ~5-8 minutes the first time (mostly the Playwright Chromium download + apt installs).

**If you already ran this before, you can skip to Step 4.**

In [ ]:
!bash setup-colab.sh

## Step 3 — Verify GPU availability

Confirms your GPU is detected and NVENC is available. If you see your GPU name + `h264_nvenc` in the encoder list, you're ready for fast GPU rendering.

In [ ]:
!nvidia-smi 2>/dev/null && echo '---' && ffmpeg -hide_banner -encoders | grep -E 'h264_nvenc|libx264' || echo 'No NVIDIA GPU — will use CPU encoder (slower but works)'

## Step 4 — Render the English video

**Full quality GPU render.** With T4 GPU + GPU capture + NVENC encoding, this takes ~8-12 minutes for a 30s reel.

The command uses:
- `--gpu` → enables Chrome GPU rasterization (faster page compositing during capture)
- `--encoder nvenc` → NVIDIA hardware H.264 encoder (5-20x faster than CPU libx264)
- `--fps 30` → capture 30 source frames per second
- `--target-fps 60` → output 60fps (frames are duplicated to reach 60)
- `--scale 1.0` → capture at full 1080×1920 resolution

**Resume tip:** If this cell fails or you interrupt it, just re-run it — it skips already-captured frames and retries the encode automatically.

In [ ]:
# Full quality GPU render (Colab T4 GPU runtime)
!node bin/forge-render scenes/digital-forge-reel-en.html \
    --output output/digital-forge-reel-en.mp4 \
    --music audio/forge_theme.wav \
    --gpu \
    --encoder nvenc \
    --fps 30 \
    --target-fps 60 \
    --scale 1.0 \
    --interpolate dup \
    --crf 18 \
    --log-level info

### Already captured frames but encode failed? Re-run encode only

If you previously ran Step 4 and the capture succeeded but the encode failed (e.g. the NVENC duplicate-audio bug from an older version), you don't need to re-capture. Just run this cell — it re-runs the encode using the frames already on disk.

In [ ]:
# Re-encode existing frames (skips capture, goes straight to encode)
# Only run this if Step 4 captured frames but failed during encoding.
!ls output/frames/ | head -3 && echo "... ($(ls output/frames/ | wc -l) frames on disk)"
!node bin/forge-render scenes/digital-forge-reel-en.html \
    --output output/digital-forge-reel-en.mp4 \
    --music audio/forge_theme.wav \
    --encoder nvenc \
    --target-fps 60 \
    --interpolate dup \
    --crf 18 \
    --log-level info

### Fast CPU fallback (no GPU needed)

If you're on a CPU-only runtime or your GPU is having issues, use this. Half-res capture + CPU encoder = ~5 min but lower quality.

In [ ]:
# Fast CPU render (no GPU) — uncomment to use
# !node bin/forge-render scenes/digital-forge-reel-en.html \
#     --output output/digital-forge-reel-en-fast.mp4 \
#     --music audio/forge_theme.wav \
#     --encoder cpu \
#     --scale 0.5 \
#     --fps 15 \
#     --target-fps 60 \
#     --interpolate dup

## Step 5 — Render the Arabic video (Algerian Darija)

Same settings as the English render. The Arabic scene uses RTL layout with Cairo + Tajawal fonts.

In [ ]:
!node bin/forge-render scenes/digital-forge-reel-ar.html \
    --output output/digital-forge-reel-ar.mp4 \
    --music audio/forge_theme.wav \
    --gpu \
    --encoder nvenc \
    --fps 30 \
    --target-fps 60 \
    --scale 1.0 \
    --interpolate dup \
    --crf 18 \
    --log-level info

## Step 6 — Verify the videos exist

Check that the MP4 files were created with the right specs (1080×1920 @ 60fps, ~30s, with audio).

In [ ]:
!ls -lh output/*.mp4 2>/dev/null || echo 'No MP4s yet'
print('---')
!ffprobe -v error -show_entries stream=codec_name,width,height,r_frame_rate,duration -of default=noprint_wrappers=1 output/digital-forge-reel-en.mp4 2>/dev/null || echo 'No English video yet'
print('---')
!ffprobe -v error -show_entries stream=codec_name,width,height,r_frame_rate,duration -of default=noprint_wrappers=1 output/digital-forge-reel-ar.mp4 2>/dev/null || echo 'No Arabic video yet'

## Step 7 — Save to Google Drive (persistent)

Colab deletes files when the session ends. Save your videos to Google Drive so they persist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/digital-forge-reel
!cp output/*.mp4 /content/drive/MyDrive/digital-forge-reel/ 2>/dev/null || echo 'No MP4s to copy yet — run Step 4 first'
!ls -lh /content/drive/MyDrive/digital-forge-reel/ 2>/dev/null

## Step 8 — Download directly to your machine

Alternative to Step 7 — downloads each MP4 directly. Useful if you don't want to mount Drive.

In [ ]:
from google.colab import files
import os

for f in os.listdir('output'):
    if f.endswith('.mp4'):
        print(f'Downloading {f}...')
        files.download(f'output/{f}')

## Troubleshooting

### "Chromium not found"
Run `!bash setup-colab.sh` again. If that fails:
```python
!npx playwright install chromium
!npx playwright install-deps chromium
```

### "Command '/usr/bin/chromium-browser' requires the chromium snap to be installed"
Playwright is picking up the snap stub. Run the dedicated fix script:
```python
!node bin/forge-fix-chromium
```
Or manually:
```python
!sudo apt remove -y chromium-browser
!npx playwright install chromium
```

### NVENC encode fails ("Option b:v cannot be applied to input url...")
**Fixed in the current version.** If you're using an older version that has this bug, update the project files. You don't need to re-capture frames — just re-run Step 4 (or the "re-encode only" cell above).

### "ffmpeg lacks h264_nvenc"
Make sure you're on a GPU runtime (Runtime → Change runtime type → T4 GPU). Then:
```python
!sudo apt install -y ffmpeg
!ffmpeg -encoders | grep h264_nvenc  # should show h264_nvenc
```
If still missing, fall back to CPU: `--encoder cpu`

### Capture is slow (~0.8 fps)
Make sure `--gpu` is in the command. Without it, Chrome uses software rendering. On T4 GPU with `--gpu`, expect ~2-4 fps (vs ~0.8 fps without).

### Video is offset / has black bars on one side
**Fixed in current version.** The renderer aligns the viewport to the stage size exactly.

### Out of memory
Use `--scale 0.5` (half-res capture) or `--fps 15` (fewer frames):
```python
!node bin/forge-render scene.html --scale 0.5 --fps 15 --output output.mp4 --music audio/forge_theme.wav
```

### Colab disconnects mid-render
Frames are saved to `output/frames/` as they're captured. Re-run the same cell — it resumes from where it left off. For extra safety, mount Drive and point `--frames-dir` at it so frames survive disconnects:
```python
!node bin/forge-render scene.html --frames-dir /content/drive/MyDrive/frames --output /content/drive/MyDrive/out.mp4
```

### Want to modify the scene
Edit `scenes/digital-forge-reel-en.html` (or `-ar.html`) directly — it's self-contained HTML, no build step. Then re-run Step 4.

### Want a completely different video format
See `docs/NEW-VIDEO-EXTENSION.md` — the system supports themes, custom scenes, and scaffolding via `node bin/forge-new my-video --theme forge`.